# Point-supervised segmentation with partial cross entropy — Colab runner

Set the runtime to a GPU first: **Runtime > Change runtime type > T4 GPU**.

The notebook clones the repository, builds the SpaceNet dataset, trains the models and
writes the figures used in `report/REPORT.md`.

In [ ]:
!nvidia-smi -L
import torch
print(torch.__version__, torch.cuda.is_available())

## 1. Code

In [ ]:
!git clone -b claude/human-code-style-tewbfy https://github.com/iMohamedMamdouh/Task1.git
%cd Task1
!pip install -q rasterio

## 2. Drive (optional)

Mounting Drive keeps the dataset and the results across sessions, so a disconnect does not
cost another download or another training run. If the mount fails (it needs third party
cookies and the popup to finish), the notebook falls back to local storage and everything
below still runs.

In [ ]:
from pathlib import Path

BACKUP = Path('/content/backup')
try:
    from google.colab import drive

    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/pointseg')
except Exception as error:
    print('continuing without Drive:', error)

BACKUP.mkdir(parents=True, exist_ok=True)
print('backup directory:', BACKUP)

## 3. Dataset

About 900 tiles from SpaceNet 1 (Rio de Janeiro), roughly 110 MB and 5-10 minutes.
If a copy is already in Drive it is restored instead of downloaded again.

In [ ]:
archive = BACKUP / 'rio.tar.gz'

if archive.exists():
    !mkdir -p data && tar xzf {archive} -C data
else:
    !python -m scripts.download_spacenet --num-tiles 900 --workers 32
    !tar czf {archive} -C data rio

!ls data/rio/images | wc -l

## 4. Check the loss

Partial CE has to reduce to `F.cross_entropy` on dense labels and to the `ignore_index`
version on sparse ones, otherwise every number below is meaningless.

In [ ]:
!python -m pytest tests -q

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

from pointseg.data import sample_points

name = Path('data/rio/tiles.txt').read_text().split()[0]
mask = (np.array(Image.open(f'data/rio/masks/{name}')) > 127).astype('uint8')
points = sample_points(mask, 5, 'balanced', np.random.default_rng(0))
print('labelled pixels:', int((points != 255).sum()), 'of', points.size)

## 5. One training run

Five labelled pixels per class per tile. On a T4 an epoch takes a few seconds, so use more
epochs than the CPU study in the report did.

In [ ]:
!python -m pointseg.train --points 5 --epochs 40 --batch-size 16 --device cuda --out runs/demo_points5.json

## 6. Full grid

Ten runs: point density 1/5/20/100 and full masks, focal gamma 0 and 2, two extra seeds.
`--workers 1` because the runs share one GPU. Finished runs are skipped, so the cell can be
re-executed after a disconnect.

In [ ]:
!python -m scripts.run_experiments --epochs 40 --batch-size 16 --device cuda --workers 1 --save-checkpoints

## 7. Figures and table

In [ ]:
!python -m scripts.make_figures --checkpoint-dir runs

from IPython.display import Image as Show, display

for name in ['annotation_example', 'label_efficiency', 'training_curves', 'qualitative']:
    path = Path('report/figures') / f'{name}.png'
    if path.exists():
        display(Show(str(path)))

In [ ]:
import pandas as pd

pd.read_csv('report/figures/results.csv')

## 8. Keep the results

In [ ]:
!cp -r runs report/figures {BACKUP}/
!ls {BACKUP}